In [1]:
import sqlite3
import csv

conn = sqlite3.connect("RH.db")
cur = conn.cursor()

In [2]:
with open("recursos_humanos.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    empleados = list(reader)

In [3]:
empleados[0]

{'satisfaction_level': '0.38',
 'last_evaluation': '0.53',
 'number_project': '2',
 'average_montly_hours': '157',
 'time_spend_company': '3',
 'Work_accident': '0',
 'left': '1',
 'promotion_last_5years': '0',
 'sales': 'sales',
 'salary': 'low'}

In [4]:
cur.execute("""
CREATE TABLE IF NOT EXISTS Detalle (
    satisfaction REAL,
    last_evaluation REAL,
    number_project INTEGER,
    average_monthly_hours INTEGER,
    time_spend_company INTEGER,
    Work_accident INTEGER,
    left INTEGER,
    promotion_last_5years INTEGER,
    sales TEXT,
    salary TEXT
);
""")

conn.commit()

In [7]:
for empleado in empleados:
    cur.execute("""
        INSERT INTO Detalle (
            satisfaction,
            last_evaluation,
            number_project,
            average_monthly_hours,
            time_spend_company,
            Work_accident,
            left,
            promotion_last_5years,
            sales,
            salary
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        empleado["satisfaction_level"],
        empleado["last_evaluation"],
        empleado["number_project"],
        empleado["average_montly_hours"],
        empleado["time_spend_company"],
        empleado["Work_accident"],
        empleado["left"],
        empleado["promotion_last_5years"],
        empleado["sales"],
        empleado["salary"]
    ))

conn.commit()

In [8]:
cur.execute("SELECT * FROM Detalle;")
result = cur.fetchall()
result[:5]

[(0.38, 0.53, 2, 157, 3, 0, 1, 0, 'sales', 'low'),
 (0.8, 0.86, 5, 262, 6, 0, 1, 0, 'sales', 'medium'),
 (0.11, 0.88, 7, 272, 4, 0, 1, 0, 'sales', 'medium'),
 (0.72, 0.87, 5, 223, 5, 0, 1, 0, 'sales', 'low'),
 (0.37, 0.52, 2, 159, 3, 0, 1, 0, 'sales', 'low')]

In [9]:
cur.execute("SELECT COUNT(*) FROM Detalle;")
cur.fetchone()

(14999,)

In [12]:
#¿Quién tiene mayor promedio de satisfacción: los que se fueron o los que se quedaron?
cur.execute("""
SELECT
    CASE
        WHEN left = 1 THEN 'Se fueron'
        WHEN left = 0 THEN 'Se quedaron'
    END AS situacion,
    AVG(satisfaction) AS promedio_satisfaccion
FROM Detalle
GROUP BY left;
""")

result = cur.fetchall()

for fila in result:
    print(fila)

('Se quedaron', 0.666809590479524)
('Se fueron', 0.44009801176141133)


In [14]:
#Promedio de horas trabajadas según salario bajo o medio
cur.execute("""
SELECT
    salary,
    AVG(average_monthly_hours) AS promedio_horas
FROM Detalle
WHERE salary IN ('low', 'medium')
GROUP BY salary;
""")

result = cur.fetchall()

for fila in result:
    print(fila)

('low', 200.9965828321487)
('medium', 201.33834936394663)


In [15]:
#Empleados promovidos en los últimos 5 años que además se fueron
cur.execute("""
SELECT *
FROM Detalle
WHERE promotion_last_5years = 1
  AND left = 1;
""")

result = cur.fetchall()

for empleado in result:
    print(empleado)

(0.45, 0.51, 2, 160, 3, 1, 1, 1, 'sales', 'low')
(0.79, 0.59, 4, 139, 3, 0, 1, 1, 'management', 'low')
(0.41, 0.46, 2, 160, 3, 0, 1, 1, 'sales', 'low')
(0.11, 0.79, 6, 292, 4, 0, 1, 1, 'technical', 'low')
(0.41, 0.56, 2, 154, 3, 0, 1, 1, 'support', 'medium')
(0.46, 0.45, 2, 138, 3, 0, 1, 1, 'IT', 'low')
(0.87, 1.0, 4, 258, 5, 1, 1, 1, 'sales', 'medium')
(0.44, 0.55, 2, 128, 3, 0, 1, 1, 'IT', 'medium')
(0.45, 0.51, 2, 160, 3, 1, 1, 1, 'sales', 'low')
(0.79, 0.59, 4, 139, 3, 0, 1, 1, 'management', 'low')
(0.41, 0.46, 2, 160, 3, 0, 1, 1, 'sales', 'low')
(0.11, 0.79, 6, 292, 4, 0, 1, 1, 'technical', 'low')
(0.41, 0.56, 2, 154, 3, 0, 1, 1, 'support', 'medium')
(0.46, 0.45, 2, 138, 3, 0, 1, 1, 'IT', 'low')
(0.45, 0.51, 2, 160, 3, 1, 1, 1, 'sales', 'low')
(0.79, 0.59, 4, 139, 3, 0, 1, 1, 'management', 'low')
(0.41, 0.46, 2, 160, 3, 0, 1, 1, 'sales', 'low')
(0.11, 0.79, 6, 292, 4, 0, 1, 1, 'technical', 'low')
(0.41, 0.56, 2, 154, 3, 0, 1, 1, 'support', 'medium')


In [16]:
#Empleados cuya última evaluación fue ≥ 0.9
cur.execute("""
SELECT *
FROM Detalle
WHERE last_evaluation >= 0.9;
""")

result = cur.fetchall()

for empleado in result:
    print(empleado)

(0.89, 1.0, 5, 224, 5, 0, 1, 0, 'sales', 'low')
(0.84, 0.92, 4, 234, 5, 0, 1, 0, 'sales', 'low')
(0.78, 0.99, 4, 255, 6, 0, 1, 0, 'sales', 'low')
(0.09, 0.95, 6, 304, 4, 0, 1, 0, 'sales', 'low')
(0.89, 0.92, 5, 242, 5, 0, 1, 0, 'sales', 'low')
(0.1, 0.94, 6, 255, 4, 0, 1, 0, 'technical', 'low')
(0.1, 0.92, 7, 307, 4, 0, 1, 0, 'support', 'low')
(0.11, 0.94, 7, 255, 4, 0, 1, 0, 'support', 'low')
(0.85, 1.0, 4, 225, 5, 0, 1, 0, 'technical', 'low')
(0.85, 0.91, 5, 226, 5, 0, 1, 0, 'management', 'medium')
(0.11, 0.93, 7, 308, 4, 0, 1, 0, 'IT', 'medium')
(0.1, 0.95, 6, 244, 5, 0, 1, 0, 'IT', 'medium')
(0.11, 0.94, 6, 286, 4, 0, 1, 0, 'IT', 'medium')
(0.9, 0.98, 4, 264, 6, 0, 1, 0, 'product_mng', 'medium')
(0.74, 0.99, 2, 277, 3, 0, 1, 0, 'IT', 'medium')
(0.11, 0.97, 6, 277, 4, 0, 1, 0, 'product_mng', 'medium')
(0.89, 1.0, 5, 246, 5, 0, 1, 0, 'sales', 'low')
(0.11, 0.97, 6, 284, 4, 0, 1, 0, 'sales', 'low')
(0.9, 1.0, 5, 221, 6, 0, 1, 0, 'sales', 'medium')
(0.09, 0.94, 7, 267, 4, 0, 1, 0, 'sal

In [17]:
#Generar archivo SQL
with open("Operaciones_RH.sql", "w", encoding="utf-8") as f:
    for line in conn.iterdump():
        f.write(line + "\n")